In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#pip install langchain_core

In [ ]:
import joblib
from langchain_core.tools import tool

In [ ]:
# Modelleri RAM'e al
kmeans = joblib.load("kmeans_ecommerce.pkl")
scaler = joblib.load("scaler.pkl")
persona_map = joblib.load("persona_map.pkl")

@tool
def get_customer_persona(age: int, tenure: int, spend: float, visits: int, return_rate: float) -> str:
    """Müşterinin davranışlarına göre K-Means modelini çalıştırıp 
    'VIP', 'İndirim Avcısı', 'Standart Kullanıcı' veya 'Uyuyan' personalarından birini döndürür."""
    
    # Veriyi DataFrame'e çeviriyoruz
    input_df = pd.DataFrame([[age, tenure, spend, visits, return_rate]], 
                            columns=['Age', 'Tenure_Months', 'Total_Spend', 'App_Visits', 'Return_Rate'])
    
    # Veriyi ölçekle ve tahmin et
    data_scaled = scaler.transform(input_df)
    cluster_id = kmeans.predict(data_scaled)[0]
    return f"Müşteri Segmenti: {persona_map[cluster_id]}"

In [ ]:
rf_model = joblib.load("rf_product_recommender.pkl")
le = joblib.load("label_encoder.pkl")
feat_imp = joblib.load("feature_importances.pkl")

@tool
def predict_product(age: int, tenure: int, spend: float, visits: int, return_rate: float, gender_kadin: int, device_mobil: int) -> str:
    """Random Forest sınıflandırma modeline göre müşterinin almayı en çok tercih edeceği ÜRÜN KATEGORİSİNİ tahmin eder."""
    
    input_df = pd.DataFrame([[age, tenure, spend, visits, return_rate, gender_kadin, device_mobil]],
                            columns=['Age', 'Tenure_Months', 'Total_Spend', 'App_Visits', 'Return_Rate', 'Gender_Kadın', 'Device_Mobil Uygulama'])
    
    # Sınıfı Tahmin Et
    pred_encoded = rf_model.predict(input_df)[0]
    predicted_product = le.inverse_transform([pred_encoded])[0]
    
    # Olasılığı (Güven Skorunu) Çıkar
    probabilities = rf_model.predict_proba(input_df)[0]
    confidence_score = max(probabilities) * 100
    
    # Modele en çok etki eden özelliği (Feature Importance) de ekle
    top_feature = feat_imp['Feature'].iloc[0]
    
    return f"Tahmin Edilen Ürün: {predicted_product}. (Model Güven Skoru: %{confidence_score:.1f}. Karar Nedeni: En çok '{top_feature}' özelliğinden etkilendim)."


In [ ]:
#pip install langchain_openai

In [ ]:
#pip install langchain_chroma

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 4 Farklı Persona için Kapsamlı Kural Kombinasyonları
kurallar = [
    """KURAL 1: 'VIP Müşteri' segmentine 'Premium Elektronik' satılacaksa; 
    'Fiyattan asla bahsetme', 'Prestij ve kalite vurgusu yap' ve koşulsuz 
    '%5 Özel Müşteri Çeki' tanımla.""",
    
    """KURAL 2: 'İndirim Avcısı' segmentine 'Gaming Ekipmanları' satılacaksa; 
    dil 'enerjik, senli benli' olmalı, 'FOMO (Kaçırma Korkusu)' yarat ve 
    '2. Ürüne %50 İndirim' ver. Ama 'İndirim Avcısı' ifadesini kullanma. """,
    
    """KURAL 3: 'Standart Kullanıcı' segmentine 'Akıllı Telefonlar' satılacaksa; 
    dil 'güven verici ve kurumsal' olmalı, cihazın hayatı nasıl kolaylaştıracağına 
    odaklan ve 'Peşin Fiyatına 6 Taksit' fırsatını öne çıkar.""",
    
    """KURAL 4: 'Uyuyan Müşteri' segmentine 'Mobil Aksesuarlar' satılacaksa; 
    'Seni Özledik' temalı nostaljik ve sıcak bir dil kullan, geri dönmesi için 
    '50 TL Hoş Geldin Kuponu' tanımla."""
]

# Metinleri sayılara çevir ve ChromaDB'ye kaydet
vectorstore = Chroma.from_texts(texts=kurallar, embedding=OpenAIEmbeddings())
# K=1 ayarı en alakalı kuralı getirir
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
@tool
def get_campaign_rules(persona: str, recommended_product: str) -> str:
    """Bulunan Müşteri Personası ve Tahmin edilen Ürün Kategorisini parametre 
    olarak alır ve bu ikiliye uygun şirket kampanya kurallarını veritabanında arar."""
    query = f"{persona} için {recommended_product} kampanya kuralı"
    docs = retriever.invoke(query) 
    # Varsa kuralı dön, yoksa LLM'in uydurmasını engellemek için net bir talimat ver
    if docs:
        return docs[0].page_content
    else:
        return "Özel bir kural bulunamadı. Lütfen standart kampanya olarak '%5 İndirim' sun ve 'kurumsal, nazik bir dil' kullan."

In [ ]:
from langchain_openai import ChatOpenAI

# Beyin (Sıcaklık 0.0, çünkü analitik karar alacak)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Alet çantamız
tools = [get_customer_persona, predict_product, get_campaign_rules]

# KİLİT ADIM: LLM'in karanlık odasına alet çantasını bırak
llm_with_tools = llm.bind_tools(tools)

In [ ]:
#pip install langgraph

In [ ]:
from langchain_core.messages import SystemMessage

system_prompt = SystemMessage(content="""Sen usta bir E-Ticaret Kampanya Yöneticisisin. 
Sana bir müşterinin verileri verilecek. LÜTFEN ŞU ADIMLARI İZLE:
1. 'get_customer_persona' aracıyla müşterinin karakterini bul.
2. 'predict_product' aracıyla müşterinin alacağı ÜRÜNÜ tahmin et. 
Cihazın sana döndüğü "%X Güven Skoru"nu ve "En Önemli Etkeni (Feature Importance)" aklında tut.
3. Bulduğun bu Persona ve Ürün bilgisini 'get_campaign_rules' aracına girerek şirketin kuralını (RAG) bul.

Görevini bitirdiğinde ÇIKTINI MUTLAKA İKİ BÖLÜM HALİNDE VER:

[YÖNETİCİ RAPORU]
Müşterinin personası, seçilen ürün, kural ve makine öğrenmesi modelinin "%X Güven Skoru" burada detaylıca yer alsın. 
Yöneticinin sistemi denetlemesini sağla.

[MÜŞTERİ MAİLİ]
Sadece müşteriye gidecek samimi, kurala uygun metni yaz. 
DİKKAT: Güven skoru veya analitik terimler ASLA bu kısma sızmamalıdır!""")

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# 1. State şemasıyla grafiği başlat
workflow = StateGraph(MessagesState)

# 2. İşçileri (Node'ları) Tanımla
def agent_node(state: MessagesState):
    # LLM kararları alır, Tools çağırır veya nihai metni yazar
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]} # State'teki mesaj listesine ekle

# 3. İşçileri haritaya ekle
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools)) # Alet çantasını çalıştıran node

In [ ]:
# Başlangıç her zaman ajana gider
workflow.add_edge(START, "agent")

# Koşullu Yönlendirme (Araç istendiyse tools'a, istenmediyse END'e git)
workflow.add_conditional_edges("agent", tools_condition)

# DÖNGÜNÜN KALBİ: Araç çalıştıktan sonra MUTLAKA Ajana dönüp rapor verir!
workflow.add_edge("tools", "agent")

# Grafiği derle ve motoru çalışmaya hazır hale getir
app = workflow.compile()

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
user_input = "Müşteri Verisi -> Yaş: 22, Tenure: 8, Spend: 400 TL, Visits: 45, İade Oranı: 0.25, Kadın: 0, Mobil: 1. Analiz et ve mail yaz."
inputs = {"messages": [system_prompt, HumanMessage(content=user_input)]}

# MOTOR ÇALIŞIYOR...
final_state = app.invoke(inputs)

# Tüm macera bittiğinde ajanın son mesajını (Mail metnini) ekrana bas
print(final_state["messages"][-1].content)

In [ ]:
user_input_2 = "Müşteri Verisi -> Yaş: 45, Tenure: 48, Spend: 5000 TL, Visits: 15, İade Oranı: 0.03, Kadın: 1, Mobil: 0. Analiz et ve mail yaz."
inputs_2 = {"messages": [system_prompt, HumanMessage(content=user_input_2)]}

# MOTOR İKİNCİ KEZ ÇALIŞIYOR...
final_state_2 = app.invoke(inputs_2)
print(final_state_2["messages"][-1].content)

In [ ]:
user_input_3 = "Müşteri Verisi -> Yaş: 58, Tenure: 70, Spend: 100 TL, Visits: 2, İade Oranı: 0.01, Kadın: 1, Mobil: 0. Analiz et ve mail yaz."
inputs_3 = {"messages": [system_prompt, HumanMessage(content=user_input_3)]}

final_state_3 = app.invoke(inputs_3)
print(final_state_3["messages"][-1].content)